# Ecuación de Onda 2D: Diferencias Finitas e Integración de Newmark

Buscamos resolver la ecuación de onda hiperbólica en un rectángulo $\Omega = [x_a, x_b] \times [y_a, y_b]$ para $t \in [0, T]$:

$$\frac{\partial^2 u}{\partial t^2} - c^2 \nabla^2 u = f(x,y,t) \quad \text{en } \Omega \times (0, T]$$

Sujeta a:
* **Condición de frontera Dirichlet:** $u(x,y,t) = g(x,y,t)$ en $\partial\Omega \times [0, T]$
* **Condición inicial de desplazamiento:** $u(x,y,0) = u_0(x,y)$ en $\Omega$
* **Condición inicial de velocidad:** $\frac{\partial u}{\partial t}(x,y,0) = v_0(x,y)$ en $\Omega$

---

### 1. Semidiscretización Espacial
Discretizando el operador $-\nabla^2$ con diferencias finitas centradas de segundo orden sobre los nodos interiores, obtenemos un sistema de ecuaciones diferenciales ordinarias de segundo orden (sistema estructural):

$$\mathbf{M}\ddot{\mathbf{u}}(t) + \mathbf{K}\mathbf{u}(t) = \mathbf{F}(t)$$

Donde:
* $\mathbf{M} = \mathbf{I}$ (matriz identidad).
* $\mathbf{K} = c^2 \left( \mathbf{D}_{xx} \otimes \mathbf{I}_y + \mathbf{I}_x \otimes \mathbf{D}_{yy} \right)$ es la matriz de rigidez discreta.
* $\mathbf{F}(t)$ contiene el término fuente $f(x_i, y_j, t)$ más los aportes de las fronteras Dirichlet.

---

### 2. Integración Temporal: Método de Newmark-$\beta$
Dado el paso temporal $\Delta t$ y los parámetros $(\beta, \gamma)$, aproximamos el estado en $t_{n+1} = (n+1)\Delta t$ mediante:

$$\mathbf{u}_{n+1} = \mathbf{u}_n + \Delta t \mathbf{v}_n + \Delta t^2 \left[ \left(\frac{1}{2} - \beta\right)\mathbf{a}_n + \beta \mathbf{a}_{n+1} \right]$$

$$\mathbf{v}_{n+1} = \mathbf{v}_n + \Delta t \left[ (1 - \gamma)\mathbf{a}_n + \gamma \mathbf{a}_{n+1} \right]$$

* **Aceleración promedio constante (Regla Trapezoidal):** $\gamma = \frac{1}{2}$, $\beta = \frac{1}{4}$. Es **incondicionalmente estable** y conserva exactamente la energía discreta (sin disipación numérica).

Despejando $\mathbf{a}_{n+1}$ en términos de $\mathbf{u}_{n+1}$ y sustituyendo en la ecuación de movimiento en $t_{n+1}$, obtenemos el sistema lineal efectivo a resolver en cada paso:

$$\underbrace{\left( \frac{1}{\beta \Delta t^2}\mathbf{I} + \mathbf{K} \right)}_{\mathbf{K}_{\text{eff}}} \mathbf{u}_{n+1} = \mathbf{F}_{n+1} + \frac{1}{\beta \Delta t^2}\mathbf{u}_n + \frac{1}{\beta \Delta t}\mathbf{v}_n + \left(\frac{1}{2\beta} - 1\right)\mathbf{a}_n$$

---

### Problema de Prueba (Modo Normal de Membrana Vibrante)
* **Dominio:** $[0, 1] \times [0, 1]$, $T = 2.0$, velocidad de onda $c = 1.0$.
* **Solución exacta:** $u(x,y,t) = \sin(\pi x)\sin(\pi y)\cos(\sqrt{2}\pi c t)$.
* **Condiciones:** $f = 0$, $g = 0$, $u_0(x,y) = \sin(\pi x)\sin(\pi y)$, $v_0(x,y) = 0$.

In [ ]:
import time
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

In [ ]:
# Dominio espacial y temporal
xa, xb = 0.0, 1.0
ya, yb = 0.0, 1.0
T = 2.0
c = 1.0  # Velocidad de propagación de la onda

# Discretización
Nx, Ny = 50, 50
Nt = 200

# Parámetros de Newmark (Aceleración promedio constante: incondicionalmente estable)
gamma = 0.5
beta = 0.25


# Funciones analíticas del problema de prueba
def u_exact(x, y, t):
  omega = np.pi * c * np.sqrt(2.0)
  return np.sin(np.pi * x) * np.sin(np.pi * y) * np.cos(omega * t)


def u0_func(x, y):
  return np.sin(np.pi * x) * np.sin(np.pi * y)


def v0_func(x, y):
  return np.zeros_like(x)


def f_source(x, y, t):
  return np.zeros_like(x)


def g_boundary(x, y, t):
  return np.zeros_like(x)

In [ ]:
def solve_wave_newmark_2d(
    xa,
    xb,
    ya,
    yb,
    T,
    c,
    Nx,
    Ny,
    Nt,
    u0_fn,
    v0_fn,
    f_fn,
    g_fn,
    beta=0.25,
    gamma=0.5,
    save_every=10,
):
  hx = (xb - xa) / Nx
  hy = (yb - ya) / Ny
  dt = T / Nt

  x = np.linspace(xa, xb, Nx + 1)
  y = np.linspace(ya, yb, Ny + 1)
  t_grid = np.linspace(0, T, Nt + 1)
  X, Y = np.meshgrid(x, y, indexing="ij")

  nx_in = Nx - 1
  ny_in = Ny - 1

  # 1. Ensamblaje de la matriz de rigidez K = c^2 * (-Laplaciano)
  diag_x = 2.0 * np.ones(nx_in) / (hx**2)
  off_x = -1.0 * np.ones(nx_in - 1) / (hx**2)
  Dxx = sp.diags([off_x, diag_x, off_x], [-1, 0, 1], format="csc")

  diag_y = 2.0 * np.ones(ny_in) / (hy**2)
  off_y = -1.0 * np.ones(ny_in - 1) / (hy**2)
  Dyy = sp.diags([off_y, diag_y, off_y], [-1, 0, 1], format="csc")

  Ix = sp.eye(nx_in, format="csc")
  Iy = sp.eye(ny_in, format="csc")
  K = (c**2) * (sp.kron(Dxx, Iy) + sp.kron(Ix, Dyy))

  # 2. Constantes de integración de Newmark
  a0 = 1.0 / (beta * dt**2)
  a1 = gamma / (beta * dt)
  a2 = 1.0 / (beta * dt)
  a3 = 1.0 / (2.0 * beta) - 1.0
  a4 = gamma / beta - 1.0
  a5 = (dt / 2.0) * (gamma / beta - 2.0)
  a6 = dt * (1.0 - gamma)
  a7 = gamma * dt

  # 3. Matriz efectiva K_eff = a0 * I + K y su factorización LU precalculada
  I_total = sp.eye(nx_in * ny_in, format="csc")
  K_eff = a0 * I_total + K
  solve_keff = spla.factorized(K_eff)  # Factorización única para máxima velocidad

  # Función auxiliar para ensamblar vector F(t) con condiciones de frontera
  def build_force_vector(t_val):
    F_mat = f_fn(X[1:-1, 1:-1], Y[1:-1, 1:-1], t_val).copy()
    # Aporte de fronteras Dirichlet en t_val
    u_left = g_fn(x[0], y[1:-1], t_val)
    u_right = g_fn(x[-1], y[1:-1], t_val)
    u_bot = g_fn(x[1:-1], y[0], t_val)
    u_top = g_fn(x[1:-1], y[-1], t_val)

    F_mat[0, :] += (c**2) * u_left / (hx**2)
    F_mat[-1, :] += (c**2) * u_right / (hx**2)
    F_mat[:, 0] += (c**2) * u_bot / (hy**2)
    F_mat[:, -1] += (c**2) * u_top / (hy**2)
    return F_mat.ravel()

  # 4. Condiciones iniciales en nodos interiores
  u_n = u0_fn(X[1:-1, 1:-1], Y[1:-1, 1:-1]).ravel()
  v_n = v0_fn(X[1:-1, 1:-1], Y[1:-1, 1:-1]).ravel()

  # Aceleración inicial: a_0 = F(0) - K * u_0
  F_0 = build_force_vector(0.0)
  a_n = F_0 - K.dot(u_n)

  # Almacenamiento de instantáneas (snapshots) y energía discreta
  snapshots = []
  snap_times = []
  energy = []

  def reconstruct_full_grid(u_vec, t_val):
    U_full = np.zeros((Nx + 1, Ny + 1))
    U_full[0, :] = g_fn(x[0], y, t_val)
    U_full[-1, :] = g_fn(x[-1], y, t_val)
    U_full[:, 0] = g_fn(x, y[0], t_val)
    U_full[:, -1] = g_fn(x, y[-1], t_val)
    U_full[1:-1, 1:-1] = u_vec.reshape((nx_in, ny_in))
    return U_full

  # Guardar estado inicial t = 0
  snapshots.append(reconstruct_full_grid(u_n, 0.0))
  snap_times.append(0.0)
  # Energía E = 0.5 * (v^T v + u^T K u) * hx * hy
  E0 = 0.5 * (np.dot(v_n, v_n) + np.dot(u_n, K.dot(u_n))) * hx * hy
  energy.append(E0)

  # 5. Bucle temporal
  for n in range(Nt):
    t_next = t_grid[n + 1]
    F_next = build_force_vector(t_next)

    # Lado derecho efectivo
    RHS = F_next + a0 * u_n + a2 * v_n + a3 * a_n

    # Resolver para u_{n+1}
    u_next = solve_keff(RHS)

    # Actualizar aceleración y velocidad
    a_next = a0 * (u_next - u_n) - a2 * v_n - a3 * a_n
    v_next = v_n + a6 * a_n + a7 * a_next

    # Avanzar estado
    u_n, v_n, a_n = u_next, v_next, a_next

    # Calcular energía discreta
    En = 0.5 * (np.dot(v_n, v_n) + np.dot(u_n, K.dot(u_n))) * hx * hy
    energy.append(En)

    if (n + 1) % save_every == 0 or (n + 1) == Nt:
      snapshots.append(reconstruct_full_grid(u_n, t_next))
      snap_times.append(t_next)

  return X, Y, t_grid, snapshots, snap_times, np.array(energy)


# Ejecutar simulación
t0 = time.time()
X, Y, t_grid, snaps, snap_t, energy = solve_wave_newmark_2d(
    xa,
    xb,
    ya,
    yb,
    T,
    c,
    Nx,
    Ny,
    Nt,
    u0_func,
    v0_func,
    f_source,
    g_boundary,
    beta=beta,
    gamma=gamma,
    save_every=25,
)
elapsed = time.time() - t0

U_final_num = snaps[-1]
U_final_ex = u_exact(X, Y, T)
err_final = np.max(np.abs(U_final_num - U_final_ex))

print(f"Tiempo de cómputo: {elapsed:.4f} s")
print(f"Error máximo en t = {T}: {err_final:.4e}")
print(
    "Variación relativa de energía:"
    f" {np.max(np.abs(energy - energy[0])) / energy[0]:.4e}"
)

In [ ]:
# 1. Visualización de instantáneas 3D en distintos tiempos
indices_to_plot = [0, 2, 4, -1]
fig = plt.figure(figsize=(16, 4))

for idx, s_idx in enumerate(indices_to_plot):
  ax = fig.add_subplot(1, 4, idx + 1, projection='3d')
  surf = ax.plot_surface(
      X,
      Y,
      snaps[s_idx],
      cmap='coolwarm',
      vmin=-1.0,
      vmax=1.0,
      edgecolor='none',
  )
  ax.set_zlim(-1.1, 1.1)
  ax.set_title(f'$t = {snap_t[s_idx]:.2f}$ s')
  ax.set_xlabel('$x$')
  ax.set_ylabel('$y$')

plt.suptitle(
    'Evolución Temporal de la Onda (Diferencias Finitas + Newmark)', y=1.02
)
plt.tight_layout()
plt.show()

# 2. Comparación de error espacial en t = T y estabilidad energética
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

err_map = np.abs(U_final_num - U_final_ex)
cf = ax1.contourf(X, Y, err_map, levels=25, cmap='viridis')
ax1.set_title(f'Error Absoluto en $t = {T}$ s (Máx: {err_final:.2e})')
ax1.set_xlabel('$x$')
ax1.set_ylabel('$y$')
fig.colorbar(cf, ax=ax1)

ax2.plot(t_grid, energy, linewidth=2, color='navy')
ax2.set_ylim(energy[0] * 0.99, energy[0] * 1.01)
ax2.set_title('Energía Total Discreta $E(t)$')
ax2.set_xlabel('Tiempo $t$ [s]')
ax2.set_ylabel('Energía')
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Refinamiento simultáneo de malla espacial (h) y temporal (dt)
resolutions = [10, 20, 40, 80]
errors_conv = []
h_vals = []

for N in resolutions:
  # Mantener relación dt / h constante
  Nt_test = 4 * N
  X_c, Y_c, _, snaps_c, _, _ = solve_wave_newmark_2d(
      xa,
      xb,
      ya,
      yb,
      T,
      c,
      N,
      N,
      Nt_test,
      u0_func,
      v0_func,
      f_source,
      g_boundary,
      beta=0.25,
      gamma=0.5,
      save_every=Nt_test,
  )
  err = np.max(np.abs(snaps_c[-1] - u_exact(X_c, Y_c, T)))
  errors_conv.append(err)
  h_vals.append((xb - xa) / N)

rates = np.log2(np.array(errors_conv[:-1]) / np.array(errors_conv[1:]))
print(
    'Órdenes de convergencia observados:',
    [round(float(r), 4) for r in rates],
)

plt.figure(figsize=(6.5, 4.5))
plt.loglog(h_vals, errors_conv, 'o-', linewidth=2, label='Error $L_\\infty(T)$')
plt.loglog(
    h_vals,
    [errors_conv[0] * (h / h_vals[0]) ** 2 for h in h_vals],
    '--',
    color='gray',
    label='Referencia $O(h^2 + \\Delta t^2)$',
)
plt.xlabel('Paso espacial $h$')
plt.ylabel('Error máximo en $t = T$')
plt.title('Convergencia del Esquema Diferencias Finitas + Newmark')
plt.grid(True, which='both', alpha=0.4)
plt.legend()
plt.show()